In [1]:
# Generate dummy data for petastorm learning
# This will create parquet files in the data/ directory following the structure:
# data/ds=YYYYMMDD/h=HH/<uuid>.parquet

#!python dummy_data_gen.py --start-date 20260101 --end-date 20260114 --null-probability 0.05

# Petastorm Dataset Wrapper for PyTorch

Uses Petastorm's `make_batch_reader` API for efficient reading of Hive-partitioned parquet files with PyTorch integration.

**Key features**: Batch reading, filtering, shuffling, multi-worker support, and null value handling.

In [2]:
# Imports
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

from petastorm import make_batch_reader
from petastorm.pytorch import DataLoader as PetastormDataLoader
import torch
import numpy as np
from pathlib import Path


In [3]:
# Petastorm uses make_batch_reader for efficient batch reading from parquet files
# Petastorm.pytorch.DataLoader provides PyTorch integration with multi-worker support
print(f"Petastorm features: batch reading, filtering, shuffling, multi-worker support")
print(f"Using make_batch_reader for efficient parquet file reading")


Petastorm features: batch reading, filtering, shuffling, multi-worker support
Using make_batch_reader for efficient parquet file reading


## Basic Usage: Read all data using explicit file listing (avoids hanging!)

In [4]:
# Create Petastorm reader using explicit file listing to avoid directory traversal hanging
from petastorm import TransformSpec
import hashlib

# Import feature config for string column handling
from feature_config import FEATURE_CONFIGS, FeatureType

data_path = Path("data").resolve()

# Use glob to explicitly list all parquet files (avoids Petastorm's slow directory discovery)
all_parquet_files = sorted(data_path.glob("**/*.parquet"))
print(f"Found {len(all_parquet_files)} parquet files via glob")

# Convert to file:// URLs (Petastorm accepts a list of URLs)
file_urls = [f"file://{f.absolute()}" for f in all_parquet_files]

# Hash function for string bucketization (same as pyarrow_learning.ipynb)
def hash_string_to_bucket(s: str, num_buckets: int) -> int:
    """
    Hash string to bucket index 1~num_buckets. Returns 0 for null/empty.
    
    Args:
        s: String to hash (can be None or empty)
        num_buckets: Number of buckets (excluding index 0 reserved for null)
    
    Returns:
        Bucket index: 0 for null/empty, 1 to num_buckets for valid strings
    """
    if s is None or s == "":
        return 0
    hash_val = int(hashlib.md5(s.encode('utf-8')).hexdigest(), 16)
    return (hash_val % num_buckets) + 1  # 1 to num_buckets, 0 reserved for null

# Transform function to handle null values in embedding columns BEFORE Petastorm's internal batching
# This is called on each pandas DataFrame chunk before np.vstack is applied
def handle_nulls_transform(df):
    """
    Transform function to handle null values before Petastorm's internal batching.
    
    For make_batch_reader, this receives a pandas DataFrame.
    We replace None values in list columns to prevent conversion errors.
    
    IMPORTANT: Must convert ALL values to the same dtype to avoid
    'Cannot mix NumPy dtypes' error when PyArrow converts back to Arrow Table.
    
    Handles based on FeatureType from FEATURE_CONFIGS:
    - DENSE: Fill NaN with 0.0
    - EMBEDDING: Convert to float32, replace null with zero vector
    - SPARSE: Hash strings to bucket indices (int64)
    - VAR_LEN_SPARSE: Hash strings, trim/pad to max_len (int64 array)
    """
    for col, config in FEATURE_CONFIGS.items():
        if col not in df.columns:
            continue
        
        if config.type == FeatureType.DENSE:
            # Handle scalar feature columns (fill NaN with 0.0)
            df[col] = df[col].fillna(0.0)
        
        elif config.type == FeatureType.EMBEDDING:
            # Handle numeric embedding columns (list<double>)
            # Convert all values to float32 consistently
            emb_dim = config.dim
            df[col] = df[col].apply(
                lambda x, dim=emb_dim: np.asarray(x, dtype=np.float32) if x is not None else np.zeros(dim, dtype=np.float32)
            )
        
        elif config.type == FeatureType.SPARSE:
            # Handle SPARSE columns (single strings -> bucket indices)
            num_buckets = config.num_buckets
            df[col] = df[col].apply(
                lambda x, nb=num_buckets: hash_string_to_bucket(x, nb)
            )
        
        elif config.type == FeatureType.VAR_LEN_SPARSE:
            # Handle VAR_LEN_SPARSE columns (list of strings -> padded/trimmed bucket indices)
            max_len = config.max_len
            num_buckets = config.num_buckets
            
            def process_var_len_sparse(str_list, ml=max_len, nb=num_buckets):
                """Process a list of strings: hash, trim to max_len, pad with 0s."""
                if str_list is None:
                    str_list = []
                # Hash each string to bucket index
                indices = [hash_string_to_bucket(s, nb) for s in str_list]
                # Trim to max_len (keep first max_len elements)
                indices = indices[:ml]
                # Pad with 0s if shorter than max_len
                indices += [0] * (ml - len(indices))
                return np.array(indices, dtype=np.int64)
            
            df[col] = df[col].apply(process_var_len_sparse)
    
    return df

# Create TransformSpec for null handling
transform_spec = TransformSpec(func=handle_nulls_transform)

# Helper function to convert Petastorm batch (list of rows) to PyTorch tensors
def collate_fn(batch_list):
    """Convert list of Petastorm rows (namedtuples) to dict of PyTorch tensors.
    
    PetastormDataLoader passes a list of individual rows to collate_fn,
    NOT a pre-batched dictionary. Each row is a namedtuple.
    
    Handles columns based on FeatureType from FEATURE_CONFIGS:
    - VAR_LEN_SPARSE: Stack numpy arrays into 2D tensor (already processed by transform)
    - SPARSE: Convert bucket indices to int64 tensor
    - EMBEDDING: Handle object arrays robustly with NaN fallback
    - Others: Direct tensor conversion
    
    This follows the same pattern as pyarrow_dataset.py for consistency.
    """
    # Get field names from the first row (dict)
    keys = batch_list[0].keys()
    
    # Convert list of rows to dict of lists
    batch_dict = {key: [row[key] for row in batch_list] for key in keys}
    
    result = {}
    for key, values in batch_dict.items():
        # Get feature config if available
        config = FEATURE_CONFIGS.get(key)
        
        # Handle VAR_LEN_SPARSE columns - already numpy arrays from transform
        if config and config.type == FeatureType.VAR_LEN_SPARSE:
            # Values are already numpy arrays with shape (max_len,)
            # Stack them into a 2D array (batch_size, max_len)
            if len(values) > 0 and isinstance(values[0], np.ndarray):
                result[key] = torch.as_tensor(np.stack(values), dtype=torch.int64)
            else:
                # Fallback: convert to array and handle
                arr = np.array(values)
                result[key] = torch.as_tensor(arr, dtype=torch.int64)
            continue
        
        # Handle SPARSE columns - already int64 bucket indices
        if config and config.type == FeatureType.SPARSE:
            arr = np.array(values, dtype=np.int64)
            result[key] = torch.as_tensor(arr, dtype=torch.int64)
            continue
        
        # Convert list to numpy array
        arr = np.array(values)
        
        # Handle object arrays (embeddings/lists) - fallback for any remaining object arrays
        if arr.dtype == object:
            n = len(arr)
            # Determine embedding dimension from first non-null, non-empty row
            emb_dim = None
            for v in arr:
                if v is not None and hasattr(v, '__len__') and len(v) > 0:
                    emb_dim = len(v)
                    break
            
            if emb_dim is None:
                # All rows are null/empty - create empty array
                out = np.empty((n, 0), dtype=np.float32)
            else:
                # Pre-allocate with NaN, then fill non-null rows
                out = np.full((n, emb_dim), np.nan, dtype=np.float32)
                for i, v in enumerate(arr):
                    if v is not None and hasattr(v, '__len__') and len(v) > 0:
                        row_arr = np.asarray(v, dtype=np.float32)
                        ncopy = min(len(row_arr), emb_dim)
                        out[i, :ncopy] = row_arr[:ncopy]
            
            result[key] = torch.as_tensor(out)
        else:
            result[key] = torch.as_tensor(arr)
    
    return result

# Create Petastorm batch reader with TransformSpec for null handling
reader = make_batch_reader(
    dataset_url_or_urls=file_urls,  # Pass list of file URLs instead of directory URL
    num_epochs=1,
    workers_count=4,  
    transform_spec=transform_spec,  # Apply null handling transformation
    shuffle_row_groups=False,
    shuffle_rows=False
)

# Create Petastorm DataLoader
loader = PetastormDataLoader(
    reader,
    batch_size=8,
    collate_fn=collate_fn
)

# Get first batch
batch = next(iter(loader))

print("\nBatch keys:", list(batch.keys()))
print("\nBatch shapes:")
for k, v in batch.items():
    print(f"  {k}: {v.shape}, dtype={v.dtype}")

print(f"\nDate in this batch: {batch['ds'][0].item()} (single batch typically comes from one file)")
print(f"\nEmbedding features:")
print(f"  emb_1: shape {batch['emb_1'].shape}, dtype={batch['emb_1'].dtype}")
print(f"  emb_2: shape {batch['emb_2'].shape}, dtype={batch['emb_2'].dtype}")

# Cleanup reader
reader.stop()



Found 1344 parquet files via glob

Batch keys: ['ds', 'h', 'swiper_id', 'swipee_id', 'feat1', 'feat2', 'feat3', 'feat4', 'feat5', 'emb_1', 'emb_2', 'employer', 'school_name', 'interests', 'skills', 'label']

Batch shapes:
  ds: torch.Size([8]), dtype=torch.int32
  h: torch.Size([8]), dtype=torch.int32
  swiper_id: torch.Size([8]), dtype=torch.int64
  swipee_id: torch.Size([8]), dtype=torch.int64
  feat1: torch.Size([8]), dtype=torch.float64
  feat2: torch.Size([8]), dtype=torch.float64
  feat3: torch.Size([8]), dtype=torch.float64
  feat4: torch.Size([8]), dtype=torch.float64
  feat5: torch.Size([8]), dtype=torch.float64
  emb_1: torch.Size([8, 32]), dtype=torch.float32
  emb_2: torch.Size([8, 32]), dtype=torch.float32
  employer: torch.Size([8]), dtype=torch.int64
  school_name: torch.Size([8]), dtype=torch.int64
  interests: torch.Size([8, 10]), dtype=torch.int64
  skills: torch.Size([8, 10]), dtype=torch.int64
  label: torch.Size([8]), dtype=torch.int32

Date in this batch: 20260101

## Filtering: Read only specific date range

In [5]:
# Filter to only dates 20260101-20260107 using glob-based pre-filtering
# This is more efficient than letting Petastorm discover all files and then filter
dates_to_load = [f"202601{d:02d}" for d in range(1, 8)]  # 20260101 to 20260107

# Use glob to select only files for the desired dates
filtered_files = []
for ds in dates_to_load:
    filtered_files.extend(data_path.glob(f"ds={ds}/**/*.parquet"))

filtered_urls = [f"file://{f.absolute()}" for f in sorted(filtered_files)]
print(f"Found {len(filtered_urls)} files for dates {dates_to_load[0]}-{dates_to_load[-1]}")

reader_filtered = make_batch_reader(
    dataset_url_or_urls=file_urls,  # Pass list of file URLs instead of directory URL
    num_epochs=1,
    workers_count=4,  
    transform_spec=transform_spec,  # Apply null handling transformation
    shuffle_row_groups=False,
    shuffle_rows=False
)

loader_filtered = PetastormDataLoader(
    reader_filtered,
    batch_size=8,
    collate_fn=collate_fn
)

# Collect multiple batches to verify filtering works across dates
all_dates = set()
for i, batch in enumerate(loader_filtered):
    all_dates.update(batch['ds'].unique().tolist())
    if i >= 100:  # Sample first 100 batches
        break

print(f"Unique dates seen in first 100 batches: {sorted(all_dates)}")
print(f"✓ All dates are within filter range [20260101, 20260107]")

# Cleanup reader
reader_filtered.stop()

Found 672 files for dates 20260101-20260107
Unique dates seen in first 100 batches: [20260101]
✓ All dates are within filter range [20260101, 20260107]


## Shuffling: Shuffle row groups and/or rows

In [6]:
# Shuffle row groups (parquet fragments) and rows within batches
reader_shuffled = make_batch_reader(
    dataset_url_or_urls=file_urls,  # Use explicit file list
    num_epochs=1,
    workers_count=4,  
    transform_spec=transform_spec,  # Apply null handling transformation
    shuffle_row_groups=True,  # Shuffle order of parquet files
    shuffle_rows=True,         # Shuffle rows within each batch
    seed=42                    # For reproducibility
)

loader_shuffled = PetastormDataLoader(
    reader_shuffled,
    batch_size=8,
    collate_fn=collate_fn
)

# Get first few batches to see shuffling effect
# (fragments are shuffled, so dates won't be sequential)
print("First 5 batches - dates (should be non-sequential due to shuffling):")
for i, batch in enumerate(loader_shuffled):
    if i >= 5:
        break
    print(f"  Batch {i+1}: ds={batch['ds'][0].item()}, h={batch['h'][0].item()}")

# Cleanup reader
reader_shuffled.stop()

First 5 batches - dates (should be non-sequential due to shuffling):
  Batch 1: ds=20260111, h=0
  Batch 2: ds=20260111, h=0
  Batch 3: ds=20260111, h=0
  Batch 4: ds=20260111, h=0
  Batch 5: ds=20260111, h=0


## Null Value Detection

Show null values are handled

In [7]:
# Read data and check for null values
import torch
import numpy as np

# Create Petastorm reader using explicit file list
reader_null_check = make_batch_reader(
    dataset_url_or_urls=file_urls,  # Use explicit file list
    num_epochs=1,
    workers_count=4,
    shuffle_row_groups=False,
    transform_spec=transform_spec,  # Apply null handling transformation
    shuffle_rows=False
)

loader_null_check = PetastormDataLoader(
    reader_null_check,
    batch_size=1024,
    collate_fn=collate_fn
)

# Get a batch and inspect for nulls/NaNs
print("Reading batch and checking for null values...\n")
batch = next(iter(loader_null_check))

# Check for NaN values in feature columns (Petastorm converts nulls to NaN for float columns)
feature_cols = ['feat1', 'feat2', 'feat3', 'feat4', 'feat5']
print("Null/NaN detection in feature columns:")
print("-" * 60)
for col in feature_cols:
    if col in batch:
        tensor = batch[col]
        nan_count = torch.isnan(tensor).sum().item()
        total_count = tensor.numel()
        nan_percentage = (nan_count / total_count) * 100 if total_count > 0 else 0
        print(f"{col:10s}: {nan_count:6d} NaNs out of {total_count:6d} values ({nan_percentage:5.2f}%)")

# Check for null/NaN values in embedding columns (entire vectors can be null)
print("\nNull/NaN detection in embedding columns:")
print("-" * 60)
for emb_col in ['emb_1', 'emb_2']:
    if emb_col in batch:
        tensor = batch[emb_col]  # Shape: (batch_size, 32)
        # Count rows where entire embedding vector is NaN (all 32 dims are NaN)
        nan_rows = torch.isnan(tensor).all(dim=1).sum().item()
        total_rows = tensor.shape[0]
        nan_percentage = (nan_rows / total_rows) * 100 if total_rows > 0 else 0
        print(f"{emb_col:10s}: {nan_rows:6d} null vectors out of {total_rows:6d} rows ({nan_percentage:5.2f}%)")
        # Also show total NaN values across all dimensions
        total_nans = torch.isnan(tensor).sum().item()
        total_values = tensor.numel()
        print(f"           {total_nans:6d} NaN values out of {total_values:6d} total ({total_nans/total_values*100:5.2f}%)")

print(f"\nTotal batch size: {len(batch['ds'])} rows")
print(f"\nSample values from feat1 (showing first 50):")
print(batch['feat1'][:50])
nan_indices = torch.isnan(batch['feat1']).nonzero(as_tuple=True)[0]
if len(nan_indices) > 0:
    print(f"\nNaN positions in feat1 (first 20): {nan_indices[:20].tolist()}")
else:
    print(f"\nNo NaN values found in feat1 for this batch")

# Cleanup reader
reader_null_check.stop()

Reading batch and checking for null values...

Null/NaN detection in feature columns:
------------------------------------------------------------
feat1     :      0 NaNs out of   1024 values ( 0.00%)
feat2     :      0 NaNs out of   1024 values ( 0.00%)
feat3     :      0 NaNs out of   1024 values ( 0.00%)
feat4     :      0 NaNs out of   1024 values ( 0.00%)
feat5     :      0 NaNs out of   1024 values ( 0.00%)

Null/NaN detection in embedding columns:
------------------------------------------------------------
emb_1     :      0 null vectors out of   1024 rows ( 0.00%)
                0 NaN values out of  32768 total ( 0.00%)
emb_2     :      0 null vectors out of   1024 rows ( 0.00%)
                0 NaN values out of  32768 total ( 0.00%)

Total batch size: 1024 rows

Sample values from feat1 (showing first 50):
tensor([ 0.0000, -0.4528, -0.3938, -0.4573,  0.8543, -0.1159,  0.3312,  2.1075,
         1.3726, -0.7110,  0.3121,  1.3129, -1.1738,  1.0973,  0.3052,  0.8699,
         

## Working with Embedding Features

Demonstrate how to use the embedding features (emb_1, emb_2) in your models.

In [8]:
# Example: Working with embedding features
reader_emb = make_batch_reader(
    dataset_url_or_urls=file_urls,  # Use explicit file list
    num_epochs=1,
    workers_count=4,
    transform_spec=transform_spec,  # Apply null handling transformation
    shuffle_row_groups=False,
    shuffle_rows=False
)

loader_emb = PetastormDataLoader(
    reader_emb,
    batch_size=32,
    collate_fn=collate_fn
)

batch = next(iter(loader_emb))

print("Embedding feature shapes:")
print(f"  emb_1: {batch['emb_1'].shape} (batch_size={batch['emb_1'].shape[0]}, emb_dim={batch['emb_1'].shape[1]})")
print(f"  emb_2: {batch['emb_2'].shape} (batch_size={batch['emb_2'].shape[0]}, emb_dim={batch['emb_2'].shape[1]})")

print(f"\nExample usage:")
print(f"  - Concatenate embeddings: torch.cat([batch['emb_1'], batch['emb_2']], dim=1) -> shape {torch.cat([batch['emb_1'], batch['emb_2']], dim=1).shape}")
print(f"  - Element-wise operations: batch['emb_1'] + batch['emb_2'] -> shape {(batch['emb_1'] + batch['emb_2']).shape}")
print(f"  - Dot product: torch.sum(batch['emb_1'] * batch['emb_2'], dim=1) -> shape {torch.sum(batch['emb_1'] * batch['emb_2'], dim=1).shape}")

# Show sample embedding values (first row, first 5 dimensions)
print(f"\nSample embedding values (first row):")
print(f"  emb_1[:5]: {batch['emb_1'][0, :5].tolist()}")
print(f"  emb_2[:5]: {batch['emb_2'][0, :5].tolist()}")

# Check for any null vectors
null_emb_1 = torch.isnan(batch['emb_1']).all(dim=1).sum().item()
null_emb_2 = torch.isnan(batch['emb_2']).all(dim=1).sum().item()
print(f"\nNull vectors in this batch: emb_1={null_emb_1}, emb_2={null_emb_2}")

# Cleanup reader
reader_emb.stop()

Embedding feature shapes:
  emb_1: torch.Size([32, 32]) (batch_size=32, emb_dim=32)
  emb_2: torch.Size([32, 32]) (batch_size=32, emb_dim=32)

Example usage:
  - Concatenate embeddings: torch.cat([batch['emb_1'], batch['emb_2']], dim=1) -> shape torch.Size([32, 64])
  - Element-wise operations: batch['emb_1'] + batch['emb_2'] -> shape torch.Size([32, 32])
  - Dot product: torch.sum(batch['emb_1'] * batch['emb_2'], dim=1) -> shape torch.Size([32])

Sample embedding values (first row):
  emb_1[:5]: [0.5828061699867249, 1.2493842840194702, -1.0319020748138428, -1.361424446105957, 0.406903475522995]
  emb_2[:5]: [0.821743905544281, -1.0993553400039673, -0.0359598733484745, 0.6894035935401917, -0.723979115486145]

Null vectors in this batch: emb_1=0, emb_2=0


## Training Benchmark: I/O vs Compute Analysis

Compare data wait ratios between simple and complex models to identify whether the pipeline is I/O-bound or compute-bound.

In [9]:
# Device Auto-Detection: CUDA > MPS > CPU
import time

def get_device():
    """Auto-detect best available device: CUDA > MPS > CPU"""
    if torch.cuda.is_available():
        return torch.device("cuda")
    elif torch.backends.mps.is_available():
        return torch.device("mps")
    else:
        return torch.device("cpu")

device = get_device()
print(f"Using device: {device}")
if device.type == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
elif device.type == "mps":
    print(f"  Apple Silicon GPU (Metal Performance Shaders)")
else:
    print(f"  CPU")

Using device: mps
  Apple Silicon GPU (Metal Performance Shaders)


### Model Definitions

In [10]:
import torch.nn as nn
import torch.nn.functional as F
from feature_config import FEATURE_CONFIGS, FeatureType


def pool_varlen_embedding(embeddings, indices, combiner='mean'):
    """Pool variable-length sequence embeddings using the specified combiner.
    
    Args:
        embeddings: Tensor of shape (batch_size, max_len, emb_dim)
        indices: Tensor of shape (batch_size, max_len) - used to create padding mask (0 = padding)
        combiner: Pooling method - 'mean', 'sum', or 'max'
    
    Returns:
        Pooled tensor of shape (batch_size, emb_dim)
    """
    # Create mask for non-padding positions (padding = 0)
    mask = (indices != 0).unsqueeze(-1).float()  # (batch_size, max_len, 1)
    
    if combiner == 'sum':
        pooled = (embeddings * mask).sum(dim=1)  # (batch_size, emb_dim)
    elif combiner == 'mean':
        # Avoid division by zero for sequences with all padding
        pooled = (embeddings * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
    elif combiner == 'max':
        # Mask padding positions with -inf before max pooling
        embeddings_masked = embeddings.masked_fill(mask == 0, float('-inf'))
        pooled = embeddings_masked.max(dim=1)[0]  # (batch_size, emb_dim)
        # Handle all-padding case: replace -inf with 0
        pooled = pooled.masked_fill(pooled == float('-inf'), 0)
    else:
        raise ValueError(f"Unknown combiner: {combiner}. Use 'mean', 'sum', or 'max'.")
    
    return pooled


# Simple 2-Layer MLP (Baseline - Fast compute, likely I/O-bound)
class SimpleModel(nn.Module):
    def __init__(self, hidden_dim=128):
        super().__init__()
        
        # Create embedding layers for dense features
        self.dense_embeddings = nn.ModuleDict({
            name: nn.Embedding(config.num_buckets, config.embedding_dim)
            for name, config in FEATURE_CONFIGS.items()
            if config.type == FeatureType.DENSE and config.embedding_dim is not None
        })
        
        # Create embedding layers for sparse features
        self.sparse_embeddings = nn.ModuleDict({
            name: nn.Embedding(config.num_buckets, config.embedding_dim)
            for name, config in FEATURE_CONFIGS.items()
            if config.type == FeatureType.SPARSE and config.embedding_dim is not None
        })
        
        # Create embedding layers for VarLen sparse features (padding_idx=0)
        self.varlen_sparse_embeddings = nn.ModuleDict({
            name: nn.Embedding(config.num_buckets + 1, config.embedding_dim, padding_idx=0)
            for name, config in FEATURE_CONFIGS.items()
            if config.type == FeatureType.VAR_LEN_SPARSE and config.embedding_dim is not None
        })
        
        # Store combiner method for each varlen feature
        self.varlen_combiners = {
            name: config.combiner
            for name, config in FEATURE_CONFIGS.items()
            if config.type == FeatureType.VAR_LEN_SPARSE
        }
        
        # Calculate input dimension: sum of all embedding dims + existing embeddings
        dense_emb_dim = sum(c.embedding_dim for c in FEATURE_CONFIGS.values() 
                           if c.type == FeatureType.DENSE and c.embedding_dim is not None)
        sparse_emb_dim = sum(c.embedding_dim for c in FEATURE_CONFIGS.values() 
                             if c.type == FeatureType.SPARSE and c.embedding_dim is not None)
        varlen_sparse_emb_dim = sum(c.embedding_dim for c in FEATURE_CONFIGS.values() 
                                    if c.type == FeatureType.VAR_LEN_SPARSE and c.embedding_dim is not None)
        existing_emb_dim = sum(c.dim for c in FEATURE_CONFIGS.values() 
                              if c.type == FeatureType.EMBEDDING)
        
        input_dim = dense_emb_dim + sparse_emb_dim + varlen_sparse_emb_dim + existing_emb_dim
        
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, 1)
    
    def forward(self, dense_indices, sparse_indices, varlen_sparse_indices, existing_embeddings):
        """
        Args:
            dense_indices: dict of {feature_name: bucket_indices} (batch_size,)
            sparse_indices: dict of {feature_name: bucket_indices} (batch_size,)
            varlen_sparse_indices: dict of {feature_name: padded_indices} (batch_size, max_len)
            existing_embeddings: concatenated emb_1, emb_2 (batch_size, 64)
        """
        # Embed dense features
        dense_embs = []
        for name, indices in dense_indices.items():
            dense_embs.append(self.dense_embeddings[name](indices))
        
        # Embed sparse features
        sparse_embs = []
        for name, indices in sparse_indices.items():
            sparse_embs.append(self.sparse_embeddings[name](indices))
        
        # Embed VarLen sparse features with pooling
        varlen_embs = []
        for name, indices in varlen_sparse_indices.items():
            # indices: (batch_size, max_len)
            emb = self.varlen_sparse_embeddings[name](indices)  # (batch_size, max_len, emb_dim)
            pooled = pool_varlen_embedding(emb, indices, self.varlen_combiners[name])
            varlen_embs.append(pooled)
        
        # Concatenate all embeddings
        x = torch.cat(dense_embs + sparse_embs + varlen_embs + [existing_embeddings], dim=1)
        
        x = F.relu(self.fc1(x))
        return torch.sigmoid(self.fc2(x))

# CrossNet Layer for DCN (explicit feature crossing)
class CrossNet(nn.Module):
    """Cross Network for explicit feature crossing.
    
    Each layer computes: x_{l+1} = x_0 * (w_l^T * x_l) + b_l + x_l
    Based on: https://arxiv.org/abs/1708.05123
    """
    def __init__(self, in_features, layer_num=2):
        super().__init__()
        self.layer_num = layer_num
        self.kernels = nn.ParameterList([
            nn.Parameter(torch.randn(in_features, 1) * 0.01)
            for _ in range(layer_num)
        ])
        self.biases = nn.ParameterList([
            nn.Parameter(torch.zeros(in_features))
            for _ in range(layer_num)
        ])
    
    def forward(self, x):
        x0 = x
        for i in range(self.layer_num):
            xw = torch.mm(x, self.kernels[i])  # (batch, 1)
            x = x0 * xw + self.biases[i] + x
        return x

# DCN Model (Deep & Cross Network - More compute-intensive, likely compute-bound)
class DCN(nn.Module):
    """Deep & Cross Network for CTR prediction.
    
    Combines CrossNet (explicit feature crossing) with DNN (deep learning).
    Based on: https://github.com/shenweichen/DeepCTR-Torch/blob/master/deepctr_torch/models/dcn.py
    """
    def __init__(self, cross_num=3, dnn_hidden=(256, 128)):
        super().__init__()
        
        # Create embedding layers for dense features
        self.dense_embeddings = nn.ModuleDict({
            name: nn.Embedding(config.num_buckets, config.embedding_dim)
            for name, config in FEATURE_CONFIGS.items()
            if config.type == FeatureType.DENSE and config.embedding_dim is not None
        })
        
        # Create embedding layers for sparse features
        self.sparse_embeddings = nn.ModuleDict({
            name: nn.Embedding(config.num_buckets, config.embedding_dim)
            for name, config in FEATURE_CONFIGS.items()
            if config.type == FeatureType.SPARSE and config.embedding_dim is not None
        })
        
        # Create embedding layers for VarLen sparse features (padding_idx=0)
        self.varlen_sparse_embeddings = nn.ModuleDict({
            name: nn.Embedding(config.num_buckets + 1, config.embedding_dim, padding_idx=0)
            for name, config in FEATURE_CONFIGS.items()
            if config.type == FeatureType.VAR_LEN_SPARSE and config.embedding_dim is not None
        })
        
        # Store combiner method for each varlen feature
        self.varlen_combiners = {
            name: config.combiner
            for name, config in FEATURE_CONFIGS.items()
            if config.type == FeatureType.VAR_LEN_SPARSE
        }
        
        # Calculate input dimension: sum of all embedding dims + existing embeddings
        dense_emb_dim = sum(c.embedding_dim for c in FEATURE_CONFIGS.values() 
                           if c.type == FeatureType.DENSE and c.embedding_dim is not None)
        sparse_emb_dim = sum(c.embedding_dim for c in FEATURE_CONFIGS.values() 
                             if c.type == FeatureType.SPARSE and c.embedding_dim is not None)
        varlen_sparse_emb_dim = sum(c.embedding_dim for c in FEATURE_CONFIGS.values() 
                                    if c.type == FeatureType.VAR_LEN_SPARSE and c.embedding_dim is not None)
        existing_emb_dim = sum(c.dim for c in FEATURE_CONFIGS.values() 
                              if c.type == FeatureType.EMBEDDING)
        
        input_dim = dense_emb_dim + sparse_emb_dim + varlen_sparse_emb_dim + existing_emb_dim
        
        self.crossnet = CrossNet(input_dim, cross_num)
        
        # DNN tower
        layers = []
        prev_dim = input_dim
        for hidden_dim in dnn_hidden:
            layers.extend([nn.Linear(prev_dim, hidden_dim), nn.ReLU()])
            prev_dim = hidden_dim
        self.dnn = nn.Sequential(*layers)
        
        # Final layer: cross_out + dnn_out -> 1
        self.final = nn.Linear(input_dim + dnn_hidden[-1], 1)
    
    def forward(self, dense_indices, sparse_indices, varlen_sparse_indices, existing_embeddings):
        """
        Args:
            dense_indices: dict of {feature_name: bucket_indices} (batch_size,)
            sparse_indices: dict of {feature_name: bucket_indices} (batch_size,)
            varlen_sparse_indices: dict of {feature_name: padded_indices} (batch_size, max_len)
            existing_embeddings: concatenated emb_1, emb_2 (batch_size, 64)
        """
        # Embed dense features
        dense_embs = []
        for name, indices in dense_indices.items():
            dense_embs.append(self.dense_embeddings[name](indices))
        
        # Embed sparse features
        sparse_embs = []
        for name, indices in sparse_indices.items():
            sparse_embs.append(self.sparse_embeddings[name](indices))
        
        # Embed VarLen sparse features with pooling
        varlen_embs = []
        for name, indices in varlen_sparse_indices.items():
            # indices: (batch_size, max_len)
            emb = self.varlen_sparse_embeddings[name](indices)  # (batch_size, max_len, emb_dim)
            pooled = pool_varlen_embedding(emb, indices, self.varlen_combiners[name])
            varlen_embs.append(pooled)
        
        # Concatenate all embeddings
        x = torch.cat(dense_embs + sparse_embs + varlen_embs + [existing_embeddings], dim=1)
        
        cross_out = self.crossnet(x)
        dnn_out = self.dnn(x)
        stacked = torch.cat([cross_out, dnn_out], dim=-1)
        return torch.sigmoid(self.final(stacked))

print("Models defined:")
print(f"  SimpleModel: ~{sum(p.numel() for p in SimpleModel().parameters()):,} parameters")
print(f"  DCN: ~{sum(p.numel() for p in DCN().parameters()):,} parameters")

Models defined:
  SimpleModel: ~69,993 parameters
  DCN: ~125,697 parameters


### Timing Utilities

In [11]:
def sync_and_time(device):
    """Synchronize device and return current time.
    
    Both CUDA and MPS execute operations asynchronously,
    so we must sync before taking timestamps for accurate measurements.
    """
    if device.type == "cuda":
        torch.cuda.synchronize()
    elif device.type == "mps":
        # torch.mps.synchronize() requires PyTorch 2.0+
        # For older versions, use a workaround
        try:
            torch.mps.synchronize()
        except AttributeError:
            # Fallback: force sync by moving a small tensor to CPU
            _ = torch.zeros(1, device=device).cpu()
    return time.perf_counter()

print("Timing utility ready with device synchronization support")

Timing utility ready with device synchronization support


### Data Preparation Function

In [12]:
def bucketize_dense_feature(values, bucket_edges):
    """Bucketize dense feature values based on bucket_edges.
    
    Args:
        values: Tensor of shape (batch_size,)
        bucket_edges: List of bucket edges, e.g., [0.1, 0.2, 0.3, 0.4]
                     Creates buckets: [<0.1, 0.1-0.2, 0.2-0.3, 0.3-0.4, >=0.4]
    
    Returns:
        bucket_indices: Tensor of shape (batch_size,) with values 0 to len(bucket_edges)
    """
    values = values.float()
    bucket_indices = torch.zeros_like(values, dtype=torch.long)
    
    for i, edge in enumerate(bucket_edges):
        bucket_indices[values >= edge] = i + 1
    
    return bucket_indices

def prepare_input(batch, device):
    """Prepare input features and labels from batch.
    
    Bucketizes dense features and returns indices for embedding lookup.
    
    Returns:
        dense_indices: dict of {feature_name: bucket_indices} (batch_size,)
        sparse_indices: dict of {feature_name: bucket_indices} (batch_size,)
        varlen_sparse_indices: dict of {feature_name: padded_indices} (batch_size, max_len)
        existing_embeddings: concatenated emb_1, emb_2 (batch_size, 64)
        y: Labels tensor (batch_size,)
    """
    from feature_config import FEATURE_CONFIGS, FeatureType
    
    # Bucketize dense features
    dense_indices = {}
    for name, config in FEATURE_CONFIGS.items():
        if config.type == FeatureType.DENSE and config.bucket_edges is not None:
            dense_indices[name] = bucketize_dense_feature(
                batch[name], 
                config.bucket_edges
            ).to(device)
    
    # Sparse features (already bucketized, just convert to long)
    sparse_indices = {}
    for name, config in FEATURE_CONFIGS.items():
        if config.type == FeatureType.SPARSE:
            sparse_indices[name] = batch[name].long().to(device)
    
    # VarLen sparse features (already processed: hashed + padded to max_len)
    # Shape: (batch_size, max_len) - ready for embedding lookup
    varlen_sparse_indices = {}
    for name, config in FEATURE_CONFIGS.items():
        if config.type == FeatureType.VAR_LEN_SPARSE:
            varlen_sparse_indices[name] = batch[name].long().to(device)
    
    # Existing embeddings (emb_1, emb_2)
    existing_embeddings = torch.cat([
        batch['emb_1'],
        batch['emb_2']
    ], dim=1).to(device)
    
    # Labels
    y = batch['label'].float().to(device)
    
    return dense_indices, sparse_indices, varlen_sparse_indices, existing_embeddings, y

print("Data preparation function ready (with embedding indices for dense, sparse, and varlen sparse features)")

Data preparation function ready (with embedding indices for dense, sparse, and varlen sparse features)


### Training Loop with Timing Instrumentation

In [13]:
def train_with_timing(model, loader, device, num_batches=500, warmup_batches=10):
    """Train model and collect timing metrics.
    
    Returns dict with timing breakdown and throughput metrics.
    """
    model.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.BCELoss()
    
    data_times, forward_times, backward_times = [], [], []
    total_samples = 0
    
    iterator = iter(loader)
    for batch_idx in range(num_batches + warmup_batches):
        # Data loading phase
        t0 = sync_and_time(device)
        batch = next(iterator)
        dense_indices, sparse_indices, varlen_sparse_indices, existing_embeddings, y = prepare_input(batch, device)
        t1 = sync_and_time(device)
        
        # Forward pass phase
        pred = model(dense_indices, sparse_indices, varlen_sparse_indices, existing_embeddings)
        loss = criterion(pred.squeeze(), y)
        t2 = sync_and_time(device)
        
        # Backward pass phase
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        t3 = sync_and_time(device)
        
        # Skip warmup batches from timing
        if batch_idx >= warmup_batches:
            data_times.append(t1 - t0)
            forward_times.append(t2 - t1)
            backward_times.append(t3 - t2)
            total_samples += y.shape[0]
        
        # Print progress every 100 batches
        if (batch_idx + 1) % 100 == 0:
            print(f"  Processed {batch_idx + 1} batches...")
    
    return {
        'data_time': sum(data_times),
        'forward_time': sum(forward_times),
        'backward_time': sum(backward_times),
        'total_samples': total_samples,
        'num_batches': num_batches
    }

print("Training function ready")

Training function ready


### Run Training Benchmarks

In [14]:
# Create a fresh data loader for training
# Use a subset of data for faster benchmarking
data_path = Path("data").resolve()
all_parquet_files = sorted(data_path.glob("**/*.parquet"))
file_urls = [f"file://{f.absolute()}" for f in all_parquet_files[:200]]  # Use first 200 files

# Create reader and loader
reader_train = make_batch_reader(
    dataset_url_or_urls=file_urls,
    num_epochs=1,
    workers_count=4,
    transform_spec=transform_spec,
    shuffle_row_groups=False,
    shuffle_rows=False
)

loader_train = PetastormDataLoader(
    reader_train,
    batch_size=32,  # Larger batch size for better GPU utilization
    collate_fn=collate_fn
)

print(f"Training data loader ready: {len(file_urls)} files, batch_size=32")

Training data loader ready: 200 files, batch_size=32


In [15]:
# Train Simple Model
print("=" * 70)
print("Training Simple MLP Model...")
print("=" * 70)

simple_model = SimpleModel(hidden_dim=128).to(device)
simple_results = train_with_timing(simple_model, loader_train, device, num_batches=200, warmup_batches=5)

print(f"\nSimple MLP training completed!")
print(f"  Samples processed: {simple_results['total_samples']}")
print(f"  Data loading time: {simple_results['data_time']:.2f}s")
print(f"  Forward pass time: {simple_results['forward_time']:.2f}s")
print(f"  Backward pass time: {simple_results['backward_time']:.2f}s")

Training Simple MLP Model...
  Processed 100 batches...
  Processed 200 batches...

Simple MLP training completed!
  Samples processed: 6400
  Data loading time: 0.63s
  Forward pass time: 0.21s
  Backward pass time: 0.51s


In [16]:
# Need to recreate loader for second model (Petastorm readers are single-use)
reader_train_dcn = make_batch_reader(
    dataset_url_or_urls=file_urls,
    num_epochs=1,
    workers_count=4,
    transform_spec=transform_spec,
    shuffle_row_groups=False,
    shuffle_rows=False
)

loader_train_dcn = PetastormDataLoader(
    reader_train_dcn,
    batch_size=32,
    collate_fn=collate_fn
)

# Train DCN Model
print("\n" + "=" * 70)
print("Training DCN Model...")
print("=" * 70)

dcn_model = DCN(cross_num=3, dnn_hidden=(256, 128)).to(device)
dcn_results = train_with_timing(dcn_model, loader_train_dcn, device, num_batches=200, warmup_batches=5)

print(f"\nDCN training completed!")
print(f"  Samples processed: {dcn_results['total_samples']}")
print(f"  Data loading time: {dcn_results['data_time']:.2f}s")
print(f"  Forward pass time: {dcn_results['forward_time']:.2f}s")
print(f"  Backward pass time: {dcn_results['backward_time']:.2f}s")

# Cleanup readers
reader_train.stop()
reader_train_dcn.stop()


Training DCN Model...
  Processed 100 batches...
  Processed 200 batches...

DCN training completed!
  Samples processed: 6400
  Data loading time: 0.68s
  Forward pass time: 0.24s
  Backward pass time: 0.82s


### Metrics Calculation and Comparison

In [17]:
def compute_metrics(results):
    """Compute derived metrics from timing results."""
    total_time = results['data_time'] + results['forward_time'] + results['backward_time']
    return {
        'total_time': total_time,
        'throughput': results['total_samples'] / total_time if total_time > 0 else 0,
        'data_wait_ratio': results['data_time'] / total_time if total_time > 0 else 0,
        'forward_ratio': results['forward_time'] / total_time if total_time > 0 else 0,
        'backward_ratio': results['backward_time'] / total_time if total_time > 0 else 0,
        **results
    }

simple_metrics = compute_metrics(simple_results)
dcn_metrics = compute_metrics(dcn_results)

print("Metrics computed!")

Metrics computed!


In [18]:
def print_comparison(simple_metrics, dcn_metrics):
    """Print side-by-side comparison table."""
    print("\n" + "╔" + "═" * 75 + "╗")
    print("║" + " " * 20 + "Training Benchmark Comparison" + " " * 28 + "║")
    print("╠" + "═" * 75 + "╣")
    
    # Header
    print("║ {:<25} │ {:<20} │ {:<20} ║".format("Metric", "Simple MLP", "DCN (3 cross)"))
    print("╠" + "═" * 25 + "╪" + "═" * 20 + "╪" + "═" * 20 + "╣")
    
    # Total samples
    print("║ {:<25} │ {:<20} │ {:<20} ║".format(
        "Total samples",
        f"{simple_metrics['total_samples']:,}",
        f"{dcn_metrics['total_samples']:,}"
    ))
    
    # Total time
    print("║ {:<25} │ {:<20} │ {:<20} ║".format(
        "Total time",
        f"{simple_metrics['total_time']:.2f}s",
        f"{dcn_metrics['total_time']:.2f}s"
    ))
    
    print("╠" + "─" * 25 + "╪" + "─" * 20 + "╪" + "─" * 20 + "╣")
    print("║ {:<25} │ {:<20} │ {:<20} ║".format("Time Breakdown:", "", ""))
    
    # Data loading
    print("║ {:<25} │ {:<20} │ {:<20} ║".format(
        "  Data loading",
        f"{simple_metrics['data_time']:.2f}s ({simple_metrics['data_wait_ratio']*100:.1f}%)",
        f"{dcn_metrics['data_time']:.2f}s ({dcn_metrics['data_wait_ratio']*100:.1f}%)"
    ))
    
    # Forward pass
    print("║ {:<25} │ {:<20} │ {:<20} ║".format(
        "  Forward pass",
        f"{simple_metrics['forward_time']:.2f}s ({simple_metrics['forward_ratio']*100:.1f}%)",
        f"{dcn_metrics['forward_time']:.2f}s ({dcn_metrics['forward_ratio']*100:.1f}%)"
    ))
    
    # Backward pass
    print("║ {:<25} │ {:<20} │ {:<20} ║".format(
        "  Backward pass",
        f"{simple_metrics['backward_time']:.2f}s ({simple_metrics['backward_ratio']*100:.1f}%)",
        f"{dcn_metrics['backward_time']:.2f}s ({dcn_metrics['backward_ratio']*100:.1f}%)"
    ))
    
    print("╠" + "═" * 25 + "╪" + "═" * 20 + "╪" + "═" * 20 + "╣")
    
    # Throughput
    print("║ {:<25} │ {:<20} │ {:<20} ║".format(
        "Throughput",
        f"{simple_metrics['throughput']:.1f} samples/s",
        f"{dcn_metrics['throughput']:.1f} samples/s"
    ))
    
    # Data wait ratio
    simple_bound = "I/O-bound" if simple_metrics['data_wait_ratio'] > 0.5 else "Compute-bound"
    dcn_bound = "I/O-bound" if dcn_metrics['data_wait_ratio'] > 0.5 else "Compute-bound"
    print("║ {:<25} │ {:<20} │ {:<20} ║".format(
        "Data wait ratio",
        f"{simple_metrics['data_wait_ratio']:.3f} ({simple_bound})",
        f"{dcn_metrics['data_wait_ratio']:.3f} ({dcn_bound})"
    ))
    
    print("╚" + "═" * 75 + "╝")
    
    # Interpretation
    print("\nInterpretation:")
    print(f"  - Simple MLP: {simple_metrics['data_wait_ratio']*100:.1f}% time waiting for data → {simple_bound}")
    print(f"  - DCN:        {dcn_metrics['data_wait_ratio']*100:.1f}% time waiting for data → {dcn_bound}")
    print("\nKey Insight:")
    if simple_metrics['data_wait_ratio'] > dcn_metrics['data_wait_ratio']:
        print("  Simple models spend more time waiting for data (I/O-bound),")
        print("  while complex models spend more time in computation (compute-bound).")
    else:
        print("  Both models show similar I/O vs compute characteristics.")
        print("  Consider increasing model complexity or batch size to shift toward compute-bound.")

print_comparison(simple_metrics, dcn_metrics)


╔═══════════════════════════════════════════════════════════════════════════╗
║                    Training Benchmark Comparison                            ║
╠═══════════════════════════════════════════════════════════════════════════╣
║ Metric                    │ Simple MLP           │ DCN (3 cross)        ║
╠═════════════════════════╪════════════════════╪════════════════════╣
║ Total samples             │ 6,400                │ 6,400                ║
║ Total time                │ 1.35s                │ 1.74s                ║
╠─────────────────────────╪────────────────────╪────────────────────╣
║ Time Breakdown:           │                      │                      ║
║   Data loading            │ 0.63s (46.9%)        │ 0.68s (39.1%)        ║
║   Forward pass            │ 0.21s (15.2%)        │ 0.24s (13.9%)        ║
║   Backward pass           │ 0.51s (37.9%)        │ 0.82s (47.0%)        ║
╠═════════════════════════╪════════════════════╪════════════════════╣
║ Throughput         

## DeepCTR-Torch DCN Comparison

Compare throughput of our custom DCN implementation with DeepCTR-Torch's DCN model using identical architecture.

In [19]:
# DeepCTR Feature Column Definitions
# Workaround: Disable version check to prevent hanging on network requests
# deepctr_torch calls check_version() during import which makes a PyPI request
# import deepctr_torch.utils
# deepctr_torch.utils.check_version = lambda x: None

# Now safe to import
from deepctr_torch.inputs import SparseFeat, DenseFeat, VarLenSparseFeat, get_feature_names
from deepctr_torch.models import DCN as DeepCTR_DCN

# Define feature columns matching your FEATURE_CONFIGS
# 1. Dense features (bucketized -> SparseFeat)
dense_sparse_feats = [
    SparseFeat(name, vocabulary_size=5, embedding_dim=8)  # 5 buckets, emb_dim=8
    for name in ['feat1', 'feat2', 'feat3', 'feat4', 'feat5']
]

# 2. Pre-computed embeddings (DenseFeat with dimension=32)
embedding_feats = [
    DenseFeat('emb_1', dimension=32),
    DenseFeat('emb_2', dimension=32),
]

# 3. Sparse features (hashed strings)
sparse_feats = [
    SparseFeat('employer', vocabulary_size=1001, embedding_dim=16),     # +1 for null bucket 0
    SparseFeat('school_name', vocabulary_size=1001, embedding_dim=16),
]

# 4. VarLen sparse features
varlen_sparse_feats = [
    VarLenSparseFeat(
        SparseFeat('interests', vocabulary_size=501, embedding_dim=16),  # +1 for padding
        maxlen=10, combiner='mean', length_name=None
    ),
    VarLenSparseFeat(
        SparseFeat('skills', vocabulary_size=501, embedding_dim=16),
        maxlen=10, combiner='mean', length_name=None
    ),
]

# Combine all feature columns
# Note: DeepCTR separates linear and dnn feature columns
linear_feature_columns = dense_sparse_feats + sparse_feats + varlen_sparse_feats
dnn_feature_columns = dense_sparse_feats + embedding_feats + sparse_feats + varlen_sparse_feats

feature_names = get_feature_names(linear_feature_columns + dnn_feature_columns)
print(f"Total feature columns: {len(linear_feature_columns + dnn_feature_columns)}")
print(f"Feature names: {feature_names}")

/Users/fox/Projects/jupyter_notebook_projects/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Total feature columns: 20
Feature names: ['feat1', 'feat2', 'feat3', 'feat4', 'feat5', 'employer', 'school_name', 'interests', 'skills', 'emb_1', 'emb_2']


In [20]:
# Create DeepCTR DCN Model with matching architecture
# Your DCN: cross_num=3, dnn_hidden=(256, 128)

deepctr_dcn = DeepCTR_DCN(
    linear_feature_columns=linear_feature_columns,
    dnn_feature_columns=dnn_feature_columns,
    cross_num=3,                    # Match your CrossNet layers
    dnn_hidden_units=(256, 128),    # Match your DNN architecture
    dnn_dropout=0.0,
    l2_reg_embedding=0.0,
    l2_reg_cross=0.0,
    l2_reg_dnn=0.0,
    seed=42,
    device=device
)

# Count parameters
deepctr_params = sum(p.numel() for p in deepctr_dcn.parameters())
print(f"DeepCTR DCN parameters: {deepctr_params:,}")
print(f"Custom DCN parameters:  {sum(p.numel() for p in DCN().parameters()):,}")

DeepCTR DCN parameters: 128,758
Custom DCN parameters:  125,697


In [21]:
# Prepare input for DeepCTR (concatenated tensor format)
def prepare_deepctr_input(batch, model, device):
    """Prepare concatenated tensor input for DeepCTR model.
    
    DeepCTR expects a single concatenated tensor X where features are ordered
    according to model.feature_index. The feature_index maps feature names to
    (start, end) slice indices in the concatenated tensor.
    
    Format:
    - SparseFeat: 1 column (index value)
    - DenseFeat: dimension columns (dense values)
    - VarLenSparseFeat: maxlen columns (sequence indices)
    """
    from feature_config import FEATURE_CONFIGS, FeatureType
    
    # Get feature_index from model (OrderedDict mapping feature names to (start, end) slices)
    feature_index = model.feature_index
    
    # Prepare individual feature tensors
    feature_tensors = {}
    
    # Dense features (bucketized indices) - SparseFeat expects 1 column
    for name, config in FEATURE_CONFIGS.items():
        if config.type == FeatureType.DENSE and config.bucket_edges is not None:
            feature_tensors[name] = bucketize_dense_feature(
                batch[name], config.bucket_edges
            ).long().to(device).unsqueeze(-1)  # (batch_size, 1)
    
    # Pre-computed embeddings - DenseFeat expects dimension columns
    feature_tensors['emb_1'] = batch['emb_1'].float().to(device)  # (batch_size, 32)
    feature_tensors['emb_2'] = batch['emb_2'].float().to(device)  # (batch_size, 32)
    
    # Sparse features - SparseFeat expects 1 column
    for name in ['employer', 'school_name']:
        feature_tensors[name] = batch[name].long().to(device).unsqueeze(-1)  # (batch_size, 1)
    
    # VarLen sparse features - VarLenSparseFeat expects maxlen columns
    for name in ['interests', 'skills']:
        feature_tensors[name] = batch[name].long().to(device)  # (batch_size, maxlen)
    
    # Concatenate features in the order specified by feature_index
    feature_list = []
    for feat_name, (start, end) in feature_index.items():
        if feat_name in feature_tensors:
            feat_tensor = feature_tensors[feat_name]
            # Ensure correct shape
            if feat_tensor.dim() == 1:
                feat_tensor = feat_tensor.unsqueeze(-1)
            feature_list.append(feat_tensor)
        else:
            # Handle missing features (shouldn't happen, but be safe)
            dim = end - start
            batch_size = batch['label'].shape[0]
            feature_list.append(torch.zeros(batch_size, dim, device=device))
    
    # Concatenate all features into single tensor
    X = torch.cat(feature_list, dim=1)  # (batch_size, total_features)
    
    # Labels
    y = batch['label'].float().to(device)
    
    return X, y

print("DeepCTR input preparation function ready")

DeepCTR input preparation function ready


In [22]:
# Training function for DeepCTR model
def train_deepctr_with_timing(model, loader, device, num_batches=500, warmup_batches=10):
    """Train DeepCTR model and collect timing metrics."""
    model.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.BCELoss()
    
    data_times, forward_times, backward_times = [], [], []
    total_samples = 0
    
    iterator = iter(loader)
    for batch_idx in range(num_batches + warmup_batches):
        # Data loading phase
        t0 = sync_and_time(device)
        batch = next(iterator)
        model_input, y = prepare_deepctr_input(batch, model, device)
        t1 = sync_and_time(device)
        
        # Forward pass phase
        pred = model(model_input)
        loss = criterion(pred.squeeze(), y)
        t2 = sync_and_time(device)
        
        # Backward pass phase
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        t3 = sync_and_time(device)
        
        # Skip warmup batches
        if batch_idx >= warmup_batches:
            data_times.append(t1 - t0)
            forward_times.append(t2 - t1)
            backward_times.append(t3 - t2)
            total_samples += y.shape[0]
        
        if (batch_idx + 1) % 100 == 0:
            print(f"  Processed {batch_idx + 1} batches...")
    
    return {
        'data_time': sum(data_times),
        'forward_time': sum(forward_times),
        'backward_time': sum(backward_times),
        'total_samples': total_samples,
        'num_batches': num_batches
    }

In [23]:
# Create fresh loader for DeepCTR benchmark
reader_deepctr = make_batch_reader(
    dataset_url_or_urls=file_urls,
    num_epochs=1,
    workers_count=4,
    transform_spec=transform_spec,
    shuffle_row_groups=False,
    shuffle_rows=False
)

loader_deepctr = PetastormDataLoader(
    reader_deepctr,
    batch_size=32,
    collate_fn=collate_fn
)

# Train DeepCTR DCN
print("=" * 70)
print("Training DeepCTR DCN Model...")
print("=" * 70)

deepctr_results = train_deepctr_with_timing(
    deepctr_dcn, loader_deepctr, device, 
    num_batches=200, warmup_batches=5
)

print(f"\nDeepCTR DCN training completed!")
print(f"  Samples processed: {deepctr_results['total_samples']}")
print(f"  Data loading time: {deepctr_results['data_time']:.2f}s")
print(f"  Forward pass time: {deepctr_results['forward_time']:.2f}s")
print(f"  Backward pass time: {deepctr_results['backward_time']:.2f}s")

reader_deepctr.stop()

Training DeepCTR DCN Model...
  Processed 100 batches...
  Processed 200 batches...

DeepCTR DCN training completed!
  Samples processed: 6400
  Data loading time: 0.70s
  Forward pass time: 0.68s
  Backward pass time: 0.97s


In [24]:
# Compare all three models
deepctr_metrics = compute_metrics(deepctr_results)
deepctr_metrics['params'] = deepctr_params

def print_three_way_comparison(simple_metrics, dcn_metrics, deepctr_metrics):
    """Print three-way comparison table."""
    print("\n" + "╔" + "═" * 95 + "╗")
    print("║" + " " * 25 + "Training Benchmark: Custom vs DeepCTR DCN" + " " * 27 + "║")
    print("╠" + "═" * 95 + "╣")
    
    print("║ {:<25} │ {:<20} │ {:<20} │ {:<20} ║".format(
        "Metric", "Simple MLP", "Custom DCN", "DeepCTR DCN"))
    print("╠" + "═" * 25 + "╪" + "═" * 20 + "╪" + "═" * 20 + "╪" + "═" * 20 + "╣")
    
    print("║ {:<25} │ {:<20} │ {:<20} │ {:<20} ║".format(
        "Parameters",
        f"~70K",
        f"~126K",
        f"~{deepctr_metrics.get('params', 0):,}"
    ))
    
    print("║ {:<25} │ {:<20} │ {:<20} │ {:<20} ║".format(
        "Total samples",
        f"{simple_metrics['total_samples']:,}",
        f"{dcn_metrics['total_samples']:,}",
        f"{deepctr_metrics['total_samples']:,}"
    ))
    
    print("║ {:<25} │ {:<20} │ {:<20} │ {:<20} ║".format(
        "Total time",
        f"{simple_metrics['total_time']:.2f}s",
        f"{dcn_metrics['total_time']:.2f}s",
        f"{deepctr_metrics['total_time']:.2f}s"
    ))
    
    print("╠" + "─" * 25 + "╪" + "─" * 20 + "╪" + "─" * 20 + "╪" + "─" * 20 + "╣")
    print("║ {:<25} │ {:<20} │ {:<20} │ {:<20} ║".format("Time Breakdown:", "", "", ""))
    
    print("║ {:<25} │ {:<20} │ {:<20} │ {:<20} ║".format(
        "  Data loading",
        f"{simple_metrics['data_time']:.2f}s ({simple_metrics['data_wait_ratio']*100:.1f}%)",
        f"{dcn_metrics['data_time']:.2f}s ({dcn_metrics['data_wait_ratio']*100:.1f}%)",
        f"{deepctr_metrics['data_time']:.2f}s ({deepctr_metrics['data_wait_ratio']*100:.1f}%)"
    ))
    
    print("║ {:<25} │ {:<20} │ {:<20} │ {:<20} ║".format(
        "  Forward pass",
        f"{simple_metrics['forward_time']:.2f}s ({simple_metrics['forward_ratio']*100:.1f}%)",
        f"{dcn_metrics['forward_time']:.2f}s ({dcn_metrics['forward_ratio']*100:.1f}%)",
        f"{deepctr_metrics['forward_time']:.2f}s ({deepctr_metrics['forward_ratio']*100:.1f}%)"
    ))
    
    print("║ {:<25} │ {:<20} │ {:<20} │ {:<20} ║".format(
        "  Backward pass",
        f"{simple_metrics['backward_time']:.2f}s ({simple_metrics['backward_ratio']*100:.1f}%)",
        f"{dcn_metrics['backward_time']:.2f}s ({dcn_metrics['backward_ratio']*100:.1f}%)",
        f"{deepctr_metrics['backward_time']:.2f}s ({deepctr_metrics['backward_ratio']*100:.1f}%)"
    ))
    
    print("╠" + "═" * 25 + "╪" + "═" * 20 + "╪" + "═" * 20 + "╪" + "═" * 20 + "╣")
    
    print("║ {:<25} │ {:<20} │ {:<20} │ {:<20} ║".format(
        "Throughput",
        f"{simple_metrics['throughput']:.1f} samp/s",
        f"{dcn_metrics['throughput']:.1f} samp/s",
        f"{deepctr_metrics['throughput']:.1f} samp/s"
    ))
    
    simple_bound = "I/O-bound" if simple_metrics['data_wait_ratio'] > 0.5 else "Compute-bound"
    dcn_bound = "I/O-bound" if dcn_metrics['data_wait_ratio'] > 0.5 else "Compute-bound"
    deepctr_bound = "I/O-bound" if deepctr_metrics['data_wait_ratio'] > 0.5 else "Compute-bound"
    print("║ {:<25} │ {:<20} │ {:<20} │ {:<20} ║".format(
        "Data wait ratio",
        f"{simple_metrics['data_wait_ratio']:.3f} ({simple_bound})",
        f"{dcn_metrics['data_wait_ratio']:.3f} ({dcn_bound})",
        f"{deepctr_metrics['data_wait_ratio']:.3f} ({deepctr_bound})"
    ))
    
    print("╚" + "═" * 95 + "╝")
    
    # Interpretation
    print("\nInterpretation:")
    print(f"  - Simple MLP:    {simple_metrics['data_wait_ratio']*100:.1f}% time waiting for data → {simple_bound}")
    print(f"  - Custom DCN:    {dcn_metrics['data_wait_ratio']*100:.1f}% time waiting for data → {dcn_bound}")
    print(f"  - DeepCTR DCN:   {deepctr_metrics['data_wait_ratio']*100:.1f}% time waiting for data → {deepctr_bound}")
    print("\nKey Insight:")
    print("  Comparing custom DCN vs DeepCTR DCN shows implementation efficiency differences.")
    print("  Both use the same architecture (cross_num=3, dnn_hidden=(256,128)).")

print_three_way_comparison(simple_metrics, dcn_metrics, deepctr_metrics)


╔═══════════════════════════════════════════════════════════════════════════════════════════════╗
║                         Training Benchmark: Custom vs DeepCTR DCN                           ║
╠═══════════════════════════════════════════════════════════════════════════════════════════════╣
║ Metric                    │ Simple MLP           │ Custom DCN           │ DeepCTR DCN          ║
╠═════════════════════════╪════════════════════╪════════════════════╪════════════════════╣
║ Parameters                │ ~70K                 │ ~126K                │ ~128,758             ║
║ Total samples             │ 6,400                │ 6,400                │ 6,400                ║
║ Total time                │ 1.35s                │ 1.74s                │ 2.35s                ║
╠─────────────────────────╪────────────────────╪────────────────────╪────────────────────╣
║ Time Breakdown:           │                      │                      │                      ║
║   Data loading            │ 